In [16]:
import os
import pandas as pd
import geopandas as gpd

base_dir = "./shapefiles"
summary_data = []

In [17]:
for folder in os.listdir(base_dir):
    folder_path = os.path.join(base_dir, folder)
    if not os.path.isdir(folder_path):
        continue

    shp_files = [f for f in os.listdir(folder_path) if f.lower().endswith(".shp")]
    if not shp_files:
        continue

    shp_path = os.path.join(folder_path, shp_files[0])
    
    try:
        gdf = gpd.read_file(shp_path)
        gdf = gdf.to_crs(epsg=3857)
        gdf['area'] = gdf.geometry.area
        gdf['Var'] = pd.to_numeric(gdf['Var'], errors='coerce').fillna(0)
        valid_gdf = gdf[gdf['Var'] > 0]
        
        total_weighted_risk = (valid_gdf['Var'] * valid_gdf['area']).sum()
        total_area = valid_gdf['area'].sum()
        
        if total_area > 0:
            risk_score = total_weighted_risk / total_area
        else:
            risk_score = 0

        summary_data.append({
            "SHP_Province": folder,
            "risk_score": round(float(risk_score), 4)
        })

        print(f"Processed {folder}: {risk_score:.2f}")

        del gdf
        
    except Exception as e:
        print(f"Error processing {folder}: {e}")

Processed Aklan: 1.76
Processed Albay: 1.74
Processed Apayao: 2.56
Processed Aurora: 2.28
Processed Bataan: 1.94
Processed Batangas: 1.82
Processed Benguet: 2.11
Processed Bohol: 1.77
Processed Bukidnon: 1.94
Processed Bulacan: 1.87
Processed Cagayan: 1.90
Processed Capiz: 1.73
Processed Catanduanes: 2.22
Processed Cavite: 1.79
Processed Cebu: 1.85
Processed Cotabato: 1.66
Processed Ifugao: 2.05
Processed Iloilo: 1.78
Processed Isabela: 2.26
Processed Kalinga: 2.23
Processed Laguna: 1.63
Processed Leyte: 1.77
Processed Maguindanao: 1.43
Processed Marinduque: 1.97
Processed Palawan: 1.86
Processed Pampanga: 1.72
Processed Pangasinan: 1.62
Processed Quezon: 1.96
Processed Quirino: 1.99
Processed Rizal: 2.01
Processed Samar: 1.77
Processed Sarangani: 2.03
Processed Sorsogon: 2.00
Processed Tarlac: 1.67
Processed Zambales: 2.06
Processed Mindoro: 1.85
Processed Camarines Sur: 1.84
Processed Metro Manila: 1.69
Processed Camarines Norte: 1.93
Processed Compostela Valley: 1.81
Processed Davao

In [18]:
province_risk = pd.DataFrame(summary_data)

#todo: this is probably bad, but ... 

def categorize(score):
    if score >  province_risk['risk_score'].quantile(2/3):
        return "High Risk"
    elif score > province_risk['risk_score'].quantile(1/3):
        return "Medium Risk"
    else:
        return "Low Risk"


In [24]:
province_risk['risk_level'] = province_risk['risk_score'].apply(categorize) # removed cause it's bad vvvv
province_risk[['SHP_Province', 'risk_score']].to_csv("province_risk_final.csv", index=False)

print(f"Exported {len(province_risk)} provinces to province_risk_final.csv")

Exported 65 provinces to province_risk_final.csv
